In [1]:
import xclone
import numpy as np
import pandas as pd
import scanpy

from collections import defaultdict

xclone.pp.efficiency_preview()

(Running XClone 0.3.8)
2025-07-22 07:55:22


/Users/elenab/Library/r-miniconda-arm64/envs/xclone/lib/python3.9/site-packages/xclone/preprocessing/_anno_data.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


[XClone efficiency] multiprocessing cpu total count in your device 8


# Load read counts
Load variant read counts and depths from the XClone package.

In [2]:
# BAF_adata = xclone.data.bch869_baf()
# tokeep_cells = BAF_adata.obs["Type"] == "Malignant"
# AD = BAF_adata.layers["AD"][tokeep_cells,:].transpose()
# DP = BAF_adata.layers["DP"][tokeep_cells,:].transpose()
# labels = BAF_adata.obs["Clone_ID"][tokeep_cells]
# print(f"AD and DP shape: {AD.shape} and {DP.shape}")
# assert AD.shape == DP.shape
# assert AD.shape[1] == len(labels)

In [3]:
labels_df = pd.read_csv("/Users/elenab/Dropbox/work/visiting/project/data/BCH869_bam/BCH869_489_clone_id.tsv", sep="\t", header=None, names=["CellID", "CloneID"])

sample_names = pd.read_csv("/Users/elenab/Dropbox/work/visiting/project/data/BCH869_xcltk/1_pileup/cellSNP.samples.tsv", sep="\t", header=None, names=["CellID"])
sample_names["CellID"] = sample_names["CellID"].map(lambda x: x.replace(".sort",""))
tokeep_cells = sample_names["CellID"].isin(labels_df["CellID"])

sample_names = sample_names.loc[sample_names["CellID"].isin(labels_df["CellID"])]

# reorder based on the cellID order in "sample_names"
labels_df = labels_df.set_index("CellID").reindex(sample_names.set_index("CellID").index).reset_index()
labels = labels_df.CloneID.tolist()
print(labels)

[1, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 2, 2, 2, 1, 2, 1, 1, 2, 2, 2, 1, 2, 1, 4, 2, 1, 2, 2, 1, 2, 1, 1, 1, 2, 2, 1, 2, 2, 1, 2, 2, 1, 2, 1, 1, 2, 2, 1, 1, 2, 1, 1, 2, 1, 2, 2, 1, 1, 1, 1, 2, 1, 2, 2, 2, 3, 1, 1, 2, 2, 2, 1, 2, 1, 3, 2, 2, 3, 2, 2, 1, 3, 2, 2, 1, 1, 2, 2, 1, 2, 2, 2, 1, 2, 1, 2, 2, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 1, 3, 2, 1, 1, 1, 3, 2, 1, 2, 1, 1, 1, 3, 1, 2, 2, 1, 1, 2, 2, 2, 1, 1, 2, 2, 1, 2, 3, 2, 1, 3, 1, 2, 1, 2, 1, 2, 1, 1, 1, 2, 1, 1, 2, 1, 1, 2, 2, 1, 2, 2, 1, 2, 3, 3, 1, 1, 2, 1, 2, 2, 1, 1, 1, 3, 1, 1, 2, 1, 1, 1, 1, 2, 1, 1, 3, 1, 2, 1, 1, 2, 1, 2, 1, 1, 2, 1, 2, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 2, 2, 1, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 1, 1, 1, 1, 3, 1, 2, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 3, 1, 3, 1, 1, 1, 1, 4, 2, 2, 1, 2, 1, 2, 2, 2, 1, 3, 2, 2, 2, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 2, 4, 3, 1, 3, 2, 3, 2, 2, 1, 2, 3, 2, 2, 2, 2, 2, 1, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 3, 2, 1, 1, 2, 1, 2, 1, 3, 2, 4, 2, 1, 2, 1, 2, 2, 3, 1, 1, 1, 2, 2, 2, 1, 4, 2, 1, 1, 3, 1, 1, 

In [4]:
AD = scanpy.read_mtx("/Users/elenab/Dropbox/work/visiting/project/data/BCH869_xcltk/1_pileup/cellSNP.tag.AD.mtx").X[:, tokeep_cells]
DP = scanpy.read_mtx("/Users/elenab/Dropbox/work/visiting/project/data/BCH869_xcltk/1_pileup/cellSNP.tag.AD.mtx").X[:, tokeep_cells]
assert AD.shape == DP.shape
print(f"AD and DP shape: {AD.shape} and {DP.shape}")
assert AD.shape[1] == len(labels)

AD and DP shape: (59728, 489) and (59728, 489)


In [5]:
row_nnz = DP.getnnz(axis=1)  # non-zero count per row
nonzero_rows = row_nnz > 5
tokeep_muts = np.where(nonzero_rows)[0]

# sample only 100 mutations
tokeep_muts = np.random.choice(tokeep_muts, size=100, replace=False)
nonzero_rows = np.zeros_like(nonzero_rows, dtype=bool)
nonzero_rows[tokeep_muts] = True
AD = AD[nonzero_rows]
DP = DP[nonzero_rows]
assert AD.shape == DP.shape
print(f"AD and DP shape: {AD.shape} and {DP.shape}")

AD and DP shape: (100, 489) and (100, 489)


# Hierarchical clustering 

In [6]:
RDR_adata = xclone.data.bch869_rdr()
gene_expression = RDR_adata.layers["raw_expr"][tokeep_cells].transpose()

In [7]:
import numpy as np
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

standardized_columns = []

for i in range(gene_expression.shape[1]):
    col = gene_expression.getcol(i).toarray().flatten()  # convert column to dense
    mean = col.mean()
    std = col.std()
    if std == 0:
        std = 1  # avoid division by zero
    standardized_col = (col - mean) / std
    standardized_columns.append(standardized_col)

X_std = np.column_stack(standardized_columns)  # shape: genes x cells

correlation_matrix = np.corrcoef(X_std.T)  # shape: cells x cells

distance_matrix = 1 - correlation_matrix
condensed_dist = squareform(distance_matrix, checks=False)

linkage_matrix = linkage(condensed_dist, method="complete")

# plt.figure(figsize=(12, 5))
# dendrogram(linkage_matrix, no_labels=True)
# plt.title("Hierarchical Clustering of Cells (Sparse Input)")
# plt.xlabel("Cell")
# plt.ylabel("1 - Pearson Correlation")
# plt.tight_layout()
# plt.show()

In [8]:
# Generate edge matrix
N = linkage_matrix.shape[0] + 1
edges = []

for i, (child1, child2, dist, _) in enumerate(linkage_matrix):
    parent = N + i

    for child in (int(child1), int(child2)):
        if child < N:
            child_height = 0.0
        else:
            child_height = linkage_matrix[child - N, 2]

        branch_length = dist - child_height
        edges.append([str(parent), str(child), branch_length])

# Compute Ornstein-Uhlenbeck kernel

First define the edge matrix of the clone phylogeny and then compute the kernel.

In [9]:
# edges = [
#     ("M", "1", 1.),
#     ("M", "2", 1.),
#     ("2", "3", 1.),
#     ("2", "4", 1.),
# ]
# clone_labels = ["1", "2", "3", "4"]
# clone_labels = labels.astype("int").astype("str").values
clone_labels = [str(i) for i in range(gene_expression.shape[1])]

In [10]:
def compute_ou_kernel_with_mrca(edges, clone_labels, lambd=1.0, sigma_squared=1.0):
    """
    OU kernel: cov_ij = sigma^2 * exp( -lambda (d_i + d_j - 2 * d_mrca) )

    Params
    - edges -> list of (parent, child, branch_length)
    - clone_labels -> list of leaf node names
    - lambd -> positive decay rate - larger value means higher local corr (going to diagonal cov)
    - sigma_squared -> trait variance at equilibrium
    """
    tree = defaultdict(list)  # keys: parents, values: (children, length)
    parent_of = {}
    for parent, child, length in edges:
        tree[parent].append((child, length))
        parent_of[child] = parent

    # identify root
    all_nodes = set(parent_of.keys()).union(tree.keys())
    root = list(all_nodes - set(parent_of.keys()))[0]

    # compute depth from root to each node
    node_depth = {}
    def dfs(node, depth):
        node_depth[node] = depth
        # iterate through the list of childred of "node"
        # update the distance from root to "node"
        for child, length in tree.get(node, []):
            dfs(child, depth + length)
    dfs(root, 0.0)

    # get paths from root to "node"
    def get_path_to_root(node):
        path = []
        while node in node_depth:
            path.append(node)
            if node == root:
                break
            node = parent_of.get(node, root)
        return path[::-1] # order from root to "node"

    ancestor_paths = {node: get_path_to_root(node) for node in clone_labels}

    def find_mrca(i, j):
        path_i = ancestor_paths[i]
        path_j = ancestor_paths[j]
        mrca = root
        for a, b in zip(path_i, path_j):
            if a == b:
                mrca = a
            else:
                break
        return mrca

    # kernel matrix
    n = len(clone_labels)
    kernel = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            node_i = clone_labels[i]
            node_j = clone_labels[j]
            d_i = node_depth[node_i]
            d_j = node_depth[node_j]
            mrca = find_mrca(node_i, node_j)
            d_mrca = node_depth[mrca]
            dist = d_i + d_j - 2 * d_mrca
            kernel[i, j] = sigma_squared * np.exp(-lambd * dist)

    return kernel

In [11]:
kernel = compute_ou_kernel_with_mrca(edges, clone_labels, lambd=0.5, sigma_squared=1.0)

assert kernel.shape[0] == kernel.shape[1] 
assert kernel.shape[0] == AD.shape[1]

print(f"OU Kernel Matrix of shape {kernel.shape}")
print(kernel)

OU Kernel Matrix of shape (489, 489)
[[1.         0.45307997 0.45307997 ... 0.45307997 0.45307997 0.53530859]
 [0.45307997 1.         0.46672734 ... 0.56439566 0.46672734 0.45307997]
 [0.45307997 0.46672734 1.         ... 0.46672734 0.6175495  0.45307997]
 ...
 [0.45307997 0.56439566 0.46672734 ... 1.         0.46672734 0.45307997]
 [0.45307997 0.46672734 0.6175495  ... 0.46672734 1.         0.45307997]
 [0.53530859 0.45307997 0.45307997 ... 0.45307997 0.45307997 1.        ]]


# Latent Gaussian Process with Binomial likelihood

In [12]:
# imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torch.autograd.functional import hessian  # compute the hessian matrix
# from scipy.special import gammaln  # to compute the binomial coefficient

In [13]:
device = torch.device("cpu")

In [14]:
# convert to tensor the data
Y = torch.sparse_coo_tensor(AD.nonzero(), AD.data, AD.shape).to_dense().to(device=device)
D = torch.sparse_coo_tensor(DP.nonzero(), DP.data, DP.shape).to_dense().to(device=device)
Z = torch.tensor(labels, dtype=torch.int, device=device)

V, N = Y.shape  # #mutations x #cells
K = len(Z.unique())  # #clones

assert Y.shape[1] == D.shape[1]
assert Y.shape[1] == Z.shape[0]
print(f"{V} SNPs, {N} cells and {K} clones\n")

print(f"Devices\nY = {Y.device}\nD = {D.device}\nZ = {Z.device}")

100 SNPs, 489 cells and 4 clones

Devices
Y = cpu
D = cpu
Z = cpu


/var/folders/nn/3msqtst97js_k0_vml6v9rd80000gn/T/ipykernel_45335/847291216.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:257.)
  Y = torch.sparse_coo_tensor(AD.nonzero(), AD.data, AD.shape).to_dense().to(device=device)


## Import auxiliary functions

In [15]:
from aux_fns import compute_laplace_term
%load_ext autoreload
%autoreload 2

## Initialize the parameters
Initialize `\mu` and the optimizer.
Note the `requires_grad=True` meaning we are optimizing it.

In [16]:
mu = [torch.zeros(N, requires_grad=True, dtype=torch.float32, device=device) for _ in range(K)]

Sigma = torch.tensor(kernel, dtype=torch.float32, device=device)
Sigma_inv = torch.linalg.inv(Sigma)  # inverese of Sigma -> N x N
log_det_Sigma = torch.logdet(Sigma + 1e-6 * torch.eye(N, device=device))  # log of determinant of Sigma

print(f"Devices\nmu = {[mu_i.device for mu_i in mu]}\nSigma = {Sigma.device}\nSigma_inv = {Sigma_inv.device}\nlog_det_Sigma = {log_det_Sigma.device}")

Devices
mu = [device(type='cpu'), device(type='cpu'), device(type='cpu'), device(type='cpu')]
Sigma = cpu
Sigma_inv = cpu
log_det_Sigma = cpu


## Optimization

In [17]:
optimizer = optim.Adam(mu, lr=1e-2)

for epoch in range(1):
    optimizer.zero_grad()  # reset the gradient

    gamma_store = torch.zeros((V, K, N))  # store all gammas
    total_laplace = 0.0
    for v in range(V):
        for k in np.unique(Z.to("cpu")):
            # compute log p(y_v | d_v, mu_v, Sigma) with laplace approx
            y_v = Y[v, :]
            d_v = D[v, :]
            laplace_term, gamma_hat = compute_laplace_term(y_v, d_v, mu[k-1], Sigma_inv, Z, k)
            total_laplace += laplace_term
            gamma_store[v, k-1] = gamma_hat

    print(f"Devices\nlaplace_term = {laplace_term.device}\ngamma_hat = {gamma_hat.device}")

    loss = -total_laplace
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch}: Loss = {loss.item():.3f}")


Devices
laplace_term = cpu
gamma_hat = cpu
Epoch 0: Loss = -82138.781


In [ ]:
# 2m 46.8s GPU (device = mps) with 100 mutations
# 22.8s CPU with 100 mutations

## Assess results

In [ ]:
VAF = Y / D
VAF[VAF.isnan()] = 0
VAF = VAF[:, Z == 1]

In [ ]:
import seaborn as sns

sns.clustermap(VAF, row_cluster=False, cmap="Blues", col_cluster=False)